# Урок 11 - Протокол агент-к-агенту (A2A)


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Что такое протокол A2A?

Протокол **Agent-to-Agent (A2A)** — это открытый стандарт, который позволяет ИИ-агентам общаться,
обнаруживать друг друга, и сотрудничать — даже если они созданы на разных фреймворках или размещены
у разных сервисов.

Ключевые концепции:

- **Обнаружение** – Агенты публикуют *Карточку агента*, которая описывает их возможности, что
  облегчает другим агентам (или оркестраторам) найти подходящего специалиста для задачи.
- **Передача сообщений** – Агенты обмениваются структурированными сообщениями через общий протокол, поэтому
  запрос от одного агента может быть понят и выполнен другим независимо от его внутренней
  реализации.
- **Жизненный цикл задачи** – A2A определяет состояния, такие как *submitted*, *working*, *completed*, и
  *failed*, предоставляя оркестратору полный обзор того, как продвигается делегированная задача.

В этом уроке мы моделируем сотрудничество в стиле A2A, подключая три специализированных туристических агента
к рабочему процессу, где каждый агент вносит свою экспертизу и передаёт результаты следующему.


## Создание специализированных туристических агентов


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Совместная работа нескольких агентов через рабочий процесс

Мы объединяем три агента в последовательный рабочий процесс, который отражает передачу сообщений A2A:

1. **CurrencyExchangeAgent** получает запрос пользователя и предоставляет рекомендации по валюте.
2. **ActivityPlannerAgent** получает обогащённый контекст и добавляет рекомендации по активностям.
3. **TravelManagerAgent** синтезирует оба входных сообщения в итоговый бриф по путешествию.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Понимание A2A в рабочей среде

В рабочей среде протокол A2A открывает мощные сценарии взаимодействия между сервисами:

| Возможность | Описание |
|---|---|
| **Взаимодействие между фреймворками** | Агент, созданный с использованием одного фреймворка, может делегировать задачи агенту, созданному с любым другим фреймворком, совместимым с A2A, обеспечивая настоящую межорганизационную совместимость. |
| **Границы сервисов** | Агенты могут находиться в отдельных микросервисах, облачных регионах или даже в разных организациях, при этом беспрепятственно сотрудничая. |
| **Динамическое обнаружение** | Оркестратор может во время выполнения запроса обращаться к реестру карточек агентов, чтобы найти наиболее подходящего специалиста для конкретной подзадачи. |
| **Потоковая передача и push-уведомления** | A2A поддерживает Server-Sent Events (SSE) для обновлений прогресса в реальном времени и push-уведомления для долго выполняющихся задач. |

Рабочий процесс, который мы построили выше, является упрощённой, выполняемой в одном процессе версией этого паттерна. В реальном
развёртывании каждый агент будет предоставлять HTTP-эндпоинт, публиковать карточку агента и общаться
через протокол A2A JSON-RPC.


## Краткое содержание

В этом уроке вы узнали:

1. **Что такое протокол A2A** — открытый стандарт для обнаружения агентов, обмена сообщениями между агентами,
   и управления задачами.
2. **Как создать специализированных агентов** — агент обмена валют, агент планирования активности,
   и оркестратор Travel Manager.
3. **Как связать агентов в рабочий процесс** — используя `WorkflowBuilder` для моделирования последовательной
   передачи сообщений между агентами.
4. **Как A2A работает в производственной среде** — обеспечивает взаимодействие между фреймворками и сервисами
   с динамическим обнаружением и потоковыми обновлениями.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Отказ от ответственности:
Этот документ был переведен с помощью сервиса переводов на базе ИИ Co-op Translator (https://github.com/Azure/co-op-translator). Хотя мы стремимся к точности, имейте в виду, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для критически важной информации рекомендуется профессиональный переводчик. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
